In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Cell 1 — Imports-
هدف: import کتابخانه‌ها

In [2]:
# Code S1 — Reproducible Leakage-Clean Pipeline
# Corresponds to manuscript Sections 2.4–2.7 and SI Text S8
# Key properties:
# - Split-first DOI grouping
# - Train-only preprocessing
# - Train-only pairwise construction
# - Test-only inference
# - Group-wise ranking evaluation


import os
import json
import math
import warnings
import numpy as np
import pandas as pd

from collections import defaultdict
from itertools import combinations

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, precision_score, recall_score, f1_score
)
from scipy.stats import binomtest

warnings.filterwarnings("ignore")

##Cell 2 — Paths and constants
هدف: تعریف مسیرها و ثابت‌ها

In [3]:
INPUT_STAGE3 = "/content/Dataset_S1.csv"
OUTPUT_DIR = "/content/rebuild_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
HOLDOUT_TEST_SIZE = 0.20
MISSINGNESS_THRESHOLD = 0.25
CORR_THRESHOLD = 0.95

ALPHA_FULL = 0.7
ALPHA_PAIR = 0.3

In [4]:
# Patch B1 — constant cell
#--------------------------------------------

RANDOM_BASELINE_N_REPEATS = 1000
RANDOM_BASELINE_SEED = 42

SANITY_HEURISTIC_PRIMARY = "pair_logp_absdiff"
SANITY_HEURISTIC_TIEBREAKERS = ["pair_tpsa_absdiff", "pair_hbond_balance_absdiff", "des_system_key"]

##Cell 3 — Load dataset

هدف: load dataset و sanity check اولیه


In [5]:
df = pd.read_csv(INPUT_STAGE3)

print(df.shape)
print(df.columns.tolist()[:30])

(757, 104)
['paper_title', 'doi', 'des_id', 'des_system_key', 'des_hba_std', 'des_hbd_std', 'molar_ratio', 'hba_canonical_component', 'hba_canonical_smiles', 'hbd_canonical_component', 'hbd_canonical_smiles', 'ratio_hba_part', 'ratio_hbd_part', 'ratio_hba_fraction', 'ratio_hbd_fraction', 'ratio_hba_to_hbd', 'analyte', 'analyte_normalized_name', 'analyte_smiles', 'cluster', 'matrix', 'technique', 'recovery', 'lod', 'ef', 'is_optimal', 'was_inferior', 'all_parse_ok', 'analyte_hba', 'analyte_hbd']


##Cell 4 — Dataset schema audit
هدف: ثبت schema و dataset card

In [6]:
schema_rows = []
for col in df.columns:
    schema_rows.append({
        "column_name": col,
        "dtype": str(df[col].dtype),
        "n_missing": int(df[col].isna().sum())
    })

schema_df = pd.DataFrame(schema_rows)
schema_df.to_csv(os.path.join(OUTPUT_DIR, "dataset_schema_summary.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "dataset_card_stage3.txt"), "w", encoding="utf-8") as f:
    f.write(f"n_rows: {df.shape[0]}\n")
    f.write(f"n_columns: {df.shape[1]}\n")
    f.write("input_dataset: stage3_modeling_table_with_descriptors_phase1.csv\n")
    f.write("used_for: split-first leakage-clean rebuild pipeline\n")

##Cell 5 — Required columns check
هدف: اطمینان از وجود ستون‌های کلیدی

In [7]:
required_cols = ["doi", "analyte", "recovery", "is_optimal"]
missing_required = [c for c in required_cols if c not in df.columns]
assert not missing_required, f"Missing required columns: {missing_required}"

if "des_system_key" not in df.columns:
    df["des_system_key"] = df.index.astype(str)

##Cell 6 — Define leakage-prone / non-feature columns
هدف: تعیین ستون‌هایی که نباید feature باشند

In [8]:
exclude_cols = {
    "doi", "analyte", "recovery", "is_optimal", "target",
    "ef", "lod", "was_inferior", "cluster", "matrix", "technique",
    "paper_title", "des_id", "des_system_key",
    "molar_ratio", "hba_canonical_component", "hbd_canonical_component",
    "analyte_normalized_name", "hba_canonical_smiles",
    "hbd_canonical_smiles", "analyte_smiles",
    "all_parse_ok", "analyte_parse_ok", "hba_parse_ok", "hbd_parse_ok",
    "ratio_parse_ok", "hba_descriptor_eligible_phase1", "hbd_descriptor_eligible_phase1"
}

exclude_cols = [c for c in df.columns if c in exclude_cols]

with open(os.path.join(OUTPUT_DIR, "leakage_excluded_columns.txt"), "w", encoding="utf-8") as f:
    for c in exclude_cols:
        f.write(c + "\n")

##Cell 7 — Define candidate feature space
هدف: ساخت feature candidateها

In [9]:
feature_candidates = [c for c in df.columns if c not in exclude_cols]
feature_candidates = [c for c in feature_candidates if pd.api.types.is_numeric_dtype(df[c])]

with open(os.path.join(OUTPUT_DIR, "full_feature_candidates_before_split.txt"), "w", encoding="utf-8") as f:
    for c in feature_candidates:
        f.write(c + "\n")

print("n_feature_candidates =", len(feature_candidates))

n_feature_candidates = 75


##Cell 8 — Split first by DOI
هدف: split first

In [10]:
gss = GroupShuffleSplit(n_splits=1, test_size=HOLDOUT_TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=df["doi"].astype(str)))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

train_dois = set(df_train["doi"].astype(str))
test_dois = set(df_test["doi"].astype(str))
overlap = train_dois.intersection(test_dois)

split_summary = pd.DataFrame([{
    "n_train_rows": len(df_train),
    "n_test_rows": len(df_test),
    "n_train_dois": len(train_dois),
    "n_test_dois": len(test_dois),
    "overlap_dois_count": len(overlap),
    "positive_rate_train": df_train["is_optimal"].mean(),
    "positive_rate_test": df_test["is_optimal"].mean(),
    "random_state": RANDOM_STATE,
    "holdout_test_size": HOLDOUT_TEST_SIZE
}])

split_summary.to_csv(os.path.join(OUTPUT_DIR, "split_audit_summary.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "train_test_doi_lists.json"), "w", encoding="utf-8") as f:
    json.dump({
        "train_dois": sorted(list(train_dois)),
        "test_dois": sorted(list(test_dois))
    }, f, ensure_ascii=False, indent=2)

##Cell 9 — Group count audit
هدف: تعداد groupهای train/test

In [11]:
# GROUPING HELPERS
#------------------

GROUP_COLS = ["doi", "analyte"]
GROUPBY_KW = dict(dropna=False)

def canonical_groupby(df):
    return df.groupby(GROUP_COLS, **GROUPBY_KW)

def canonical_group_size(df):
    return canonical_groupby(df).size().reset_index(name="n")

In [12]:
train_groups = canonical_group_size(df_train)
test_groups = canonical_group_size(df_test)

pd.DataFrame([{
    "n_train_groups": len(train_groups),
    "n_test_groups": len(test_groups)
}]).to_csv(os.path.join(OUTPUT_DIR, "train_test_group_counts.csv"), index=False)

train_groups.to_csv(os.path.join(OUTPUT_DIR, "train_groups_detailed.csv"), index=False)
test_groups.to_csv(os.path.join(OUTPUT_DIR, "test_groups_detailed.csv"), index=False)

##Cell 10 — Training-only feature filtering
هدف: missingness / constant / correlation روی train only

In [13]:
audit_rows = []

X_train_raw = df_train[feature_candidates].copy()
X_test_raw = df_test[feature_candidates].copy()

# missingness
keep_after_missing = []
for c in X_train_raw.columns:
    miss = X_train_raw[c].isna().mean()
    if miss <= MISSINGNESS_THRESHOLD:
        keep_after_missing.append(c)
        audit_rows.append({"feature": c, "missing_fraction_train": miss, "drop_reason": "keep_after_missingness", "retained_final": None})
    else:
        audit_rows.append({"feature": c, "missing_fraction_train": miss, "drop_reason": "drop_high_missingness", "retained_final": False})

X_train_1 = X_train_raw[keep_after_missing].copy()
X_test_1 = X_test_raw[keep_after_missing].copy()

# constant
keep_after_constant = []
for c in X_train_1.columns:
    nunique = X_train_1[c].nunique(dropna=True)
    if nunique > 1:
        keep_after_constant.append(c)
    else:
        for row in audit_rows:
            if row["feature"] == c and row["retained_final"] is None:
                row["drop_reason"] = "drop_constant"
                row["retained_final"] = False

X_train_2 = X_train_1[keep_after_constant].copy()
X_test_2 = X_test_1[keep_after_constant].copy()

# correlation
corr = X_train_2.corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]

selected_features = [c for c in X_train_2.columns if c not in to_drop_corr]

for row in audit_rows:
    if row["feature"] in selected_features and row["retained_final"] is None:
        row["retained_final"] = True
    elif row["feature"] in to_drop_corr and row["retained_final"] is None:
        row["drop_reason"] = "drop_high_correlation"
        row["retained_final"] = False

feature_audit_df = pd.DataFrame(audit_rows)
feature_audit_df.to_csv(os.path.join(OUTPUT_DIR, "feature_filtering_audit.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "train_selected_features_after_filtering.txt"), "w", encoding="utf-8") as f:
    for c in selected_features:
        f.write(c + "\n")

##Cell 11 — Build clean train/test feature matrices

In [14]:
X_train = df_train[selected_features].copy()
X_test = df_test[selected_features].copy()

y_train = df_train["is_optimal"].astype(int).values
y_test = df_test["is_optimal"].astype(int).values

##Cell 12 — Define informative groups on train only
هدف: train-only informative groups

In [15]:
def summarize_groups(df_in):
    rows = []

    for group_key, g in canonical_groupby(df_in):
        doi, analyte = group_key
        g_original = g.copy()

        n_rows_raw = len(g_original)
        n_missing_recovery = int(g_original["recovery"].isna().sum())

        g = g_original.dropna(subset=["recovery"]).copy()
        n_rows_after_recovery_filter = len(g)

        if len(g) == 0:
            rows.append({
                "doi": doi,
                "analyte": analyte,
                "group_size_raw": n_rows_raw,
                "group_size_used": 0,
                "n_missing_recovery": n_missing_recovery,
                "best_recovery": np.nan,
                "second_best_recovery": np.nan,
                "delta_recovery": np.nan,
                "n_positive": 0,
                "has_both_classes": 0,
                "informative_flag": 0,
                "group_excluded_reason": "all_recovery_missing"
            })
            continue

        vals = g["recovery"].astype(float).sort_values(ascending=False).tolist()
        best = vals[0]
        second = vals[1] if len(vals) > 1 else np.nan
        delta = best - second if len(vals) > 1 else np.nan

        n_positive = int(g["is_optimal"].sum())
        has_both_classes = int(g["is_optimal"].nunique() > 1)
        informative_flag = int((len(g) >= 3) and has_both_classes)

        if len(g) < 3:
            reason = "group_size_lt_3"
        elif not has_both_classes:
            reason = "single_class_only"
        else:
            reason = "informative"

        rows.append({
            "doi": doi,
            "analyte": analyte,
            "group_size_raw": n_rows_raw,
            "group_size_used": n_rows_after_recovery_filter,
            "n_missing_recovery": n_missing_recovery,
            "best_recovery": best,
            "second_best_recovery": second,
            "delta_recovery": delta,
            "n_positive": n_positive,
            "has_both_classes": has_both_classes,
            "informative_flag": informative_flag,
            "group_excluded_reason": reason
        })

    out = pd.DataFrame(rows)

    if len(out):
        out["analyte_is_missing"] = out["analyte"].isna().astype(int)

    return out

train_group_summary = summarize_groups(df_train)
test_group_summary = summarize_groups(df_test)

train_group_summary.to_csv(os.path.join(OUTPUT_DIR, "train_informative_groups.csv"), index=False)
test_group_summary.to_csv(os.path.join(OUTPUT_DIR, "test_informative_groups.csv"), index=False)

train_group_summary.groupby(["informative_flag"], dropna=False).size().reset_index(name="n").to_csv(
    os.path.join(OUTPUT_DIR, "train_informative_groups_summary.csv"), index=False
)

test_group_summary.groupby(["informative_flag"], dropna=False).size().reset_index(name="n").to_csv(
    os.path.join(OUTPUT_DIR, "test_informative_groups_summary.csv"), index=False
)

pd.DataFrame([{
    "n_train_groups_from_counts_file": len(train_groups),
    "n_train_groups_from_summary_file": len(train_group_summary),
    "n_test_groups_from_counts_file": len(test_groups),
    "n_test_groups_from_summary_file": len(test_group_summary),
    "train_group_count_match": int(len(train_groups) == len(train_group_summary)),
    "test_group_count_match": int(len(test_groups) == len(test_group_summary))
}]).to_csv(os.path.join(OUTPUT_DIR, "phase5_reconciliation_audit.csv"), index=False)

train_group_keys_counts = set(map(tuple, train_groups[GROUP_COLS].to_records(index=False)))
train_group_keys_summary = set(map(tuple, train_group_summary[GROUP_COLS].to_records(index=False)))

test_group_keys_counts = set(map(tuple, test_groups[GROUP_COLS].to_records(index=False)))
test_group_keys_summary = set(map(tuple, test_group_summary[GROUP_COLS].to_records(index=False)))

missing_from_train_summary = train_group_keys_counts - train_group_keys_summary
missing_from_test_summary = test_group_keys_counts - test_group_keys_summary

pd.DataFrame(list(missing_from_train_summary), columns=GROUP_COLS).to_csv(
    os.path.join(OUTPUT_DIR, "train_groups_missing_from_summary.csv"), index=False
)

pd.DataFrame(list(missing_from_test_summary), columns=GROUP_COLS).to_csv(
    os.path.join(OUTPUT_DIR, "test_groups_missing_from_summary.csv"), index=False
)

pd.DataFrame([{
    "n_missing_from_train_summary": len(missing_from_train_summary),
    "n_missing_from_test_summary": len(missing_from_test_summary)
}]).to_csv(os.path.join(OUTPUT_DIR, "phase5_missing_group_key_audit.csv"), index=False)

##Cell 13 — Build pairwise dataset from train only
هدف: بازسازی clean pairwise train dataset

In [16]:
assert len(train_group_summary) == len(train_groups), "Phase 5 reconciliation failed: train group counts still mismatch."
assert len(test_group_summary) == len(test_groups), "Phase 5 reconciliation failed: test group counts still mismatch."

pair_rows = []

train_inf_keys = set(
    train_group_summary.loc[train_group_summary["informative_flag"] == 1, ["doi", "analyte"]]
    .apply(tuple, axis=1)
)

for (doi, analyte), g in canonical_groupby(df_train):
    if (doi, analyte) not in train_inf_keys:
        continue
    g = g.copy().reset_index(drop=True)
    for i, j in combinations(range(len(g)), 2):
        yi = int(g.loc[i, "is_optimal"])
        yj = int(g.loc[j, "is_optimal"])
        if yi == yj:
            continue

        fi = g.loc[i, selected_features].values.astype(float)
        fj = g.loc[j, selected_features].values.astype(float)

        pair_rows.append({
            "doi": doi,
            "analyte": analyte,
            "candidate_i_id": str(g.loc[i, "des_system_key"]),
            "candidate_j_id": str(g.loc[j, "des_system_key"]),
            "pair_label": int(yi > yj),
            **{f"diff__{feat}": fi[k] - fj[k] for k, feat in enumerate(selected_features)}
        })
        pair_rows.append({
            "doi": doi,
            "analyte": analyte,
            "candidate_i_id": str(g.loc[j, "des_system_key"]),
            "candidate_j_id": str(g.loc[i, "des_system_key"]),
            "pair_label": int(yj > yi),
            **{f"diff__{feat}": fj[k] - fi[k] for k, feat in enumerate(selected_features)}
        })

pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(os.path.join(OUTPUT_DIR, "train_pairwise_dataset.csv"), index=False)

pair_features = [c for c in pair_df.columns if c.startswith("diff__")]
pd.DataFrame({"feature": pair_features}).to_csv(
    os.path.join(OUTPUT_DIR, "train_pairwise_allowed_feature_list.csv"), index=False
)

pairs_per_group = pair_df.groupby(GROUP_COLS, dropna=False).size().reset_index(name="n_directional_pairs")

pd.DataFrame([{
    "total_pairwise_rows": len(pair_df),
    "unique_informative_groups": pairs_per_group.shape[0] if len(pairs_per_group) else 0,
    "median_pairs_per_group": float(pairs_per_group["n_directional_pairs"].median()) if len(pairs_per_group) else 0,
    "min_pairs_per_group": int(pairs_per_group["n_directional_pairs"].min()) if len(pairs_per_group) else 0,
    "max_pairs_per_group": int(pairs_per_group["n_directional_pairs"].max()) if len(pairs_per_group) else 0,
    "effective_unordered_pairs_est": int(len(pair_df) / 2),
    "bidirectional_used": True
}]).to_csv(os.path.join(OUTPUT_DIR, "train_pairwise_dataset_summary.csv"), index=False)

with open(os.path.join(OUTPUT_DIR, "pairwise_dependency_audit.txt"), "w", encoding="utf-8") as f:
    f.write("Pairwise dataset built only from df_train after Phase 5 reconciliation.\n")
    f.write(f"Train group summary rows: {len(train_group_summary)}\n")
    f.write(f"Train informative groups: {len(train_inf_keys)}\n")
    f.write(f"Pairwise rows: {len(pair_df)}\n")

##Cell 14 — Train full model

In [17]:
full_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        solver="lbfgs",
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

full_model.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=42))])

##Cell 15 — Train pairwise model

In [18]:
X_pair = pair_df[pair_features].copy()
y_pair = pair_df["pair_label"].astype(int).values

pair_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        solver="lbfgs",
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

pair_model.fit(X_pair, y_pair)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=42))])

##Cell 16 — Export model feature lists

In [19]:
pd.DataFrame({"feature": selected_features}).to_csv(
    os.path.join(OUTPUT_DIR, "full_model_feature_list.csv"), index=False
)
pd.DataFrame({"feature": pair_features}).to_csv(
    os.path.join(OUTPUT_DIR, "pairwise_model_feature_list.csv"), index=False
)

##Cell 17 — Holdout classification predictions

In [20]:
full_probs = full_model.predict_proba(X_test)[:, 1]
full_preds = (full_probs >= 0.5).astype(int)

holdout_pred_df = df_test[["doi", "analyte", "des_system_key", "is_optimal"]].copy()
holdout_pred_df["y_pred_prob_full"] = full_probs
holdout_pred_df["y_pred_label_full"] = full_preds

holdout_pred_df.to_csv(os.path.join(OUTPUT_DIR, "full_model_holdout_predictions.csv"), index=False)

##Cell 18 — Pairwise inference inside held-out groups

In [21]:
pairwise_comp_rows = []
baseline_rank_rows = []
hybrid_rank_rows = []

for (doi, analyte), g in canonical_groupby(df_test):
    g = g.copy().reset_index(drop=True)
    g["full_score"] = full_model.predict_proba(g[selected_features])[:, 1]
    g["pairwise_aggregate_score"] = 0.0

    n = len(g)
    if n >= 2:
        for i, j in combinations(range(n), 2):
            fi = g.loc[i, selected_features].values.astype(float)
            fj = g.loc[j, selected_features].values.astype(float)
            diff_ij = pd.DataFrame([{f"diff__{feat}": fi[k] - fj[k] for k, feat in enumerate(selected_features)}])
            p_ij = pair_model.predict_proba(diff_ij[pair_features])[:, 1][0]

            pairwise_comp_rows.append({
                "doi": doi,
                "analyte": analyte,
                "candidate_i_id": str(g.loc[i, "des_system_key"]),
                "candidate_j_id": str(g.loc[j, "des_system_key"]),
                "p_ij": p_ij,
                "predicted_preference": int(p_ij >= 0.5)
            })

            g.loc[i, "pairwise_aggregate_score"] += p_ij
            g.loc[j, "pairwise_aggregate_score"] += (1 - p_ij)

        g["pairwise_aggregate_score"] = g["pairwise_aggregate_score"] / (n - 1)

    g["hybrid_score"] = ALPHA_FULL * g["full_score"] + ALPHA_PAIR * g["pairwise_aggregate_score"]
    g["baseline_rank"] = g["full_score"].rank(method="first", ascending=False)
    g["hybrid_rank"] = g["hybrid_score"].rank(method="first", ascending=False)

    for _, row in g.iterrows():
        baseline_rank_rows.append({
            "doi": doi,
            "analyte": analyte,
            "des_system_key": row["des_system_key"],
            "is_optimal": row["is_optimal"],
            "score_baseline": row["full_score"],
            "rank_baseline": row["baseline_rank"]
        })

        hybrid_rank_rows.append({
            "doi": doi,
            "analyte": analyte,
            "des_system_key": row["des_system_key"],
            "is_optimal": row["is_optimal"],
            "score_full": row["full_score"],
            "score_pairwise": row["pairwise_aggregate_score"],
            "score_hybrid": row["hybrid_score"],
            "rank_hybrid": row["hybrid_rank"]
        })

pairwise_comp_df = pd.DataFrame(pairwise_comp_rows)
pairwise_comp_df.to_csv(os.path.join(OUTPUT_DIR, "pairwise_model_holdout_comparisons.csv"), index=False)

pd.DataFrame(baseline_rank_rows).to_csv(
    os.path.join(OUTPUT_DIR, "baseline_group_ranking_results.csv"), index=False
)
pd.DataFrame(hybrid_rank_rows).to_csv(
    os.path.join(OUTPUT_DIR, "hybrid_group_ranking_results.csv"), index=False
)

##Cell 19 — Classification metrics

In [22]:
metrics_df = pd.DataFrame([
    {
        "model_name": "baseline_logreg_full_clean",
        "ROC_AUC": roc_auc_score(y_test, full_probs),
        "PR_AUC": average_precision_score(y_test, full_probs),
        "MCC": matthews_corrcoef(y_test, full_preds),
        "balanced_accuracy": balanced_accuracy_score(y_test, full_preds),
        "precision": precision_score(y_test, full_preds),
        "recall": recall_score(y_test, full_preds),
        "F1": f1_score(y_test, full_preds)
    }
])

# hybrid row-level classification can optionally be left out unless explicitly defined row-wise
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "holdout_classification_metrics.csv"), index=False)

##Cell 20 — Group ranking metrics

In [23]:
base_df = pd.DataFrame(baseline_rank_rows)
hyb_df = pd.DataFrame(hybrid_rank_rows)

def compute_group_metrics(rank_df, rank_col):
    rows = []
    for (doi, analyte), g in rank_df.groupby(GROUP_COLS, dropna=False):
        g = g.sort_values(rank_col, ascending=True).copy()
        n = len(g)
        opt_positions = g.loc[g["is_optimal"] == 1, rank_col].tolist()
        if len(opt_positions) == 0:
            continue
        best_rank = min(opt_positions)
        rows.append({
            "doi": doi,
            "analyte": analyte,
            "group_size": n,
            "Hit@1": int(best_rank == 1),
            "Hit@3": int(best_rank <= 3),
            "MRR": 1.0 / best_rank
        })
    return pd.DataFrame(rows)

base_group = compute_group_metrics(base_df, "rank_baseline")
hyb_group = compute_group_metrics(hyb_df, "rank_hybrid")

In [24]:

#Patch B2 — helper function cell
# ------------------------------

def rank_groups_from_df(rank_df, score_col, ascending=False, out_rank_col="rank"):
    rows = []
    for (doi, analyte), g in rank_df.groupby(GROUP_COLS, dropna=False):
        g = g.copy()
        g[out_rank_col] = g[score_col].rank(method="first", ascending=ascending)
        rows.append(g)
    return pd.concat(rows, ignore_index=True)

def compute_group_metrics_from_ranked_df(rank_df, rank_col):
    rows = []
    for (doi, analyte), g in rank_df.groupby(GROUP_COLS, dropna=False):
        g = g.sort_values(rank_col, ascending=True).copy()
        n = len(g)
        opt_positions = g.loc[g["is_optimal"] == 1, rank_col].tolist()
        if len(opt_positions) == 0:
            continue
        best_rank = min(opt_positions)
        rows.append({
            "doi": doi,
            "analyte": analyte,
            "group_size": n,
            "Hit@1": int(best_rank == 1),
            "Hit@3": int(best_rank <= 3),
            "MRR": 1.0 / best_rank
        })
    return pd.DataFrame(rows)

In [25]:
# Patch B3 — ساخت heuristic baseline ranking results
#---------------------------------------------------------

heur_rows = []

for (doi, analyte), g in canonical_groupby(df_test):
    g = g.copy().reset_index(drop=True)

    # deterministic heuristic score:
    # lower pair_logp_absdiff is better
    # tie-breakers: lower pair_tpsa_absdiff, lower pair_hbond_balance_absdiff, then des_system_key
    sort_cols = ["pair_logp_absdiff", "pair_tpsa_absdiff", "pair_hbond_balance_absdiff", "des_system_key"]
    g = g.sort_values(sort_cols, ascending=[True, True, True, True]).reset_index(drop=True)
    g["rank_heuristic"] = np.arange(1, len(g) + 1)

    for _, row in g.iterrows():
        heur_rows.append({
            "doi": doi,
            "analyte": analyte,
            "des_system_key": row["des_system_key"],
            "is_optimal": row["is_optimal"],
            "heuristic_primary_score": row["pair_logp_absdiff"],
            "heuristic_tpsa_score": row["pair_tpsa_absdiff"],
            "heuristic_hbond_balance_score": row["pair_hbond_balance_absdiff"],
            "rank_heuristic": row["rank_heuristic"]
        })

heur_df = pd.DataFrame(heur_rows)
heur_df.to_csv(os.path.join(OUTPUT_DIR, "heuristic_group_ranking_results.csv"), index=False)

heur_group = compute_group_metrics_from_ranked_df(heur_df, "rank_heuristic")

In [26]:
# Patch B4 — ساخت random ranking baseline
# ---------------------------------------

rng = np.random.default_rng(RANDOM_BASELINE_SEED)

random_rep_summaries = []
random_rank_rows_all = []

for rep in range(RANDOM_BASELINE_N_REPEATS):
    rep_rows = []

    for (doi, analyte), g in canonical_groupby(df_test):
        g = g.copy().reset_index(drop=True)

        perm = rng.permutation(len(g))
        g = g.iloc[perm].reset_index(drop=True)
        g["rank_random"] = np.arange(1, len(g) + 1)

        for _, row in g.iterrows():
            rep_rows.append({
                "repeat_id": rep,
                "doi": doi,
                "analyte": analyte,
                "des_system_key": row["des_system_key"],
                "is_optimal": row["is_optimal"],
                "rank_random": row["rank_random"]
            })

    rep_df = pd.DataFrame(rep_rows)
    random_rank_rows_all.append(rep_df)

    rep_group = compute_group_metrics_from_ranked_df(rep_df, "rank_random")

    random_rep_summaries.append({
        "repeat_id": rep,
        "all_Hit@1": rep_group["Hit@1"].mean(),
        "all_Hit@3": rep_group["Hit@3"].mean(),
        "all_MRR": rep_group["MRR"].mean(),
        "ge2_Hit@1": rep_group.loc[rep_group["group_size"] >= 2, "Hit@1"].mean(),
        "ge2_Hit@3": rep_group.loc[rep_group["group_size"] >= 2, "Hit@3"].mean(),
        "ge2_MRR": rep_group.loc[rep_group["group_size"] >= 2, "MRR"].mean(),
        "ge3_Hit@1": rep_group.loc[rep_group["group_size"] >= 3, "Hit@1"].mean(),
        "ge3_Hit@3": rep_group.loc[rep_group["group_size"] >= 3, "Hit@3"].mean(),
        "ge3_MRR": rep_group.loc[rep_group["group_size"] >= 3, "MRR"].mean()
    })

random_rank_df = pd.concat(random_rank_rows_all, ignore_index=True)
random_rank_df.to_csv(os.path.join(OUTPUT_DIR, "random_group_ranking_results_all_repeats.csv"), index=False)

random_summary_df = pd.DataFrame(random_rep_summaries)
random_summary_df.to_csv(os.path.join(OUTPUT_DIR, "random_group_ranking_repeat_summary.csv"), index=False)

In [27]:
# Patch B5 — ساخت جدول metricهای sanity baseline
# ------------------------------------------------

def summarize_single_group_metric_df(df_group, label):
    return {
        "scope": label,
        "Hit@1": df_group["Hit@1"].mean(),
        "Hit@3": df_group["Hit@3"].mean(),
        "MRR": df_group["MRR"].mean(),
        "n_groups": len(df_group)
    }

heur_all = summarize_single_group_metric_df(heur_group, "all_groups")
heur_ge2 = summarize_single_group_metric_df(heur_group[heur_group["group_size"] >= 2], "comparable_ge_2")
heur_ge3 = summarize_single_group_metric_df(heur_group[heur_group["group_size"] >= 3], "comparable_ge_3")

rand_metrics = pd.DataFrame([
    {
        "scope": "all_groups",
        "Hit@1_mean": random_summary_df["all_Hit@1"].mean(),
        "Hit@1_sd": random_summary_df["all_Hit@1"].std(),
        "Hit@3_mean": random_summary_df["all_Hit@3"].mean(),
        "Hit@3_sd": random_summary_df["all_Hit@3"].std(),
        "MRR_mean": random_summary_df["all_MRR"].mean(),
        "MRR_sd": random_summary_df["all_MRR"].std()
    },
    {
        "scope": "comparable_ge_2",
        "Hit@1_mean": random_summary_df["ge2_Hit@1"].mean(),
        "Hit@1_sd": random_summary_df["ge2_Hit@1"].std(),
        "Hit@3_mean": random_summary_df["ge2_Hit@3"].mean(),
        "Hit@3_sd": random_summary_df["ge2_Hit@3"].std(),
        "MRR_mean": random_summary_df["ge2_MRR"].mean(),
        "MRR_sd": random_summary_df["ge2_MRR"].std()
    },
    {
        "scope": "comparable_ge_3",
        "Hit@1_mean": random_summary_df["ge3_Hit@1"].mean(),
        "Hit@1_sd": random_summary_df["ge3_Hit@1"].std(),
        "Hit@3_mean": random_summary_df["ge3_Hit@3"].mean(),
        "Hit@3_sd": random_summary_df["ge3_Hit@3"].std(),
        "MRR_mean": random_summary_df["ge3_MRR"].mean(),
        "MRR_sd": random_summary_df["ge3_MRR"].std()
    }
])

heur_metrics = pd.DataFrame([heur_all, heur_ge2, heur_ge3])
heur_metrics.to_csv(os.path.join(OUTPUT_DIR, "heuristic_group_ranking_metrics.csv"), index=False)
rand_metrics.to_csv(os.path.join(OUTPUT_DIR, "random_group_ranking_metrics.csv"), index=False)

##Cell 21 — Merge group comparison and scopes

In [28]:
group_comp = base_group.merge(
    hyb_group,
    on=["doi", "analyte", "group_size"],
    suffixes=("_baseline", "_hybrid")
)

def summarize_scope(df_scope, label):
    better = int((df_scope["MRR_hybrid"] > df_scope["MRR_baseline"]).sum())
    worse = int((df_scope["MRR_hybrid"] < df_scope["MRR_baseline"]).sum())
    equal = int((df_scope["MRR_hybrid"] == df_scope["MRR_baseline"]).sum())

    return {
        "scope": label,
        "n_groups": len(df_scope),
        "baseline_Hit@1": df_scope["Hit@1_baseline"].mean(),
        "hybrid_Hit@1": df_scope["Hit@1_hybrid"].mean(),
        "baseline_Hit@3": df_scope["Hit@3_baseline"].mean(),
        "hybrid_Hit@3": df_scope["Hit@3_hybrid"].mean(),
        "baseline_MRR": df_scope["MRR_baseline"].mean(),
        "hybrid_MRR": df_scope["MRR_hybrid"].mean(),
        "hybrid_better": better,
        "hybrid_worse": worse,
        "equal": equal
    }

all_scope = summarize_scope(group_comp, "all_groups")
ge2_scope = summarize_scope(group_comp[group_comp["group_size"] >= 2], "comparable_ge_2")
ge3_scope = summarize_scope(group_comp[group_comp["group_size"] >= 3], "comparable_ge_3")

pd.DataFrame([all_scope, ge2_scope, ge3_scope]).to_csv(
    os.path.join(OUTPUT_DIR, "group_ranking_metrics_all_and_comparable.csv"), index=False
)

In [29]:

# Patch B6 — ساخت فایل مقایسه نهایی sanity vs baseline vs hybrid
#-----------------------------------------------------------------

main_metrics = pd.read_csv(os.path.join(OUTPUT_DIR, "group_ranking_metrics_all_and_comparable.csv"))

main_metrics_slim = main_metrics[[
    "scope",
    "baseline_Hit@1", "hybrid_Hit@1",
    "baseline_Hit@3", "hybrid_Hit@3",
    "baseline_MRR", "hybrid_MRR",
    "n_groups"
]].copy()

heur_metrics_renamed = heur_metrics.rename(columns={
    "Hit@1": "heuristic_Hit@1",
    "Hit@3": "heuristic_Hit@3",
    "MRR": "heuristic_MRR"
})[["scope", "heuristic_Hit@1", "heuristic_Hit@3", "heuristic_MRR"]]

rand_metrics_renamed = rand_metrics.rename(columns={
    "Hit@1_mean": "random_Hit@1_mean",
    "Hit@1_sd": "random_Hit@1_sd",
    "Hit@3_mean": "random_Hit@3_mean",
    "Hit@3_sd": "random_Hit@3_sd",
    "MRR_mean": "random_MRR_mean",
    "MRR_sd": "random_MRR_sd"
})

sanity_compare = (
    main_metrics_slim
    .merge(heur_metrics_renamed, on="scope", how="left")
    .merge(rand_metrics_renamed, on="scope", how="left")
)

sanity_compare.to_csv(os.path.join(OUTPUT_DIR, "sanity_baseline_comparison.csv"), index=False)

##Cell 22 — Uncertainty estimates
این cell را می‌توان ساده نگه داشت یا بعداً refine کرد. فعلاً structure لازم را تولید کند.

In [30]:
def bootstrap_ci(x, n_boot=2000, random_state=42):
    rng = np.random.default_rng(random_state)
    vals = np.array(x, dtype=float)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boots.append(sample.mean())
    return np.quantile(boots, 0.025), np.quantile(boots, 0.975)

unc_rows = []
for label, df_scope in {
    "all_groups": group_comp,
    "comparable_ge_2": group_comp[group_comp["group_size"] >= 2],
    "comparable_ge_3": group_comp[group_comp["group_size"] >= 3]
}.items():
    for model_name, hit_col, mrr_col in [
        ("Baseline", "Hit@1_baseline", "MRR_baseline"),
        ("Hybrid", "Hit@1_hybrid", "MRR_hybrid")
    ]:
        hit_mean = df_scope[hit_col].mean()
        mrr_mean = df_scope[mrr_col].mean()
        mrr_low, mrr_high = bootstrap_ci(df_scope[mrr_col].values)

        # ساده‌سازی: CI ویلسون برای Hit@1 را بعداً اگر لازم شد دقیق‌تر می‌کنیم
        unc_rows.append({
            "scope": label,
            "model": model_name,
            "Hit@1": hit_mean,
            "Hit@1_CI_low": np.nan,
            "Hit@1_CI_high": np.nan,
            "MRR": mrr_mean,
            "MRR_CI_low": mrr_low,
            "MRR_CI_high": mrr_high
        })

pd.DataFrame(unc_rows).to_csv(os.path.join(OUTPUT_DIR, "group_ranking_uncertainty.csv"), index=False)

##Cell 23 — Exact sign test

In [31]:
stat_rows = []

for label, df_scope in {
    "comparable_ge_2": group_comp[group_comp["group_size"] >= 2],
    "comparable_ge_3": group_comp[group_comp["group_size"] >= 3]
}.items():
    better = int((df_scope["MRR_hybrid"] > df_scope["MRR_baseline"]).sum())
    worse = int((df_scope["MRR_hybrid"] < df_scope["MRR_baseline"]).sum())
    n = better + worse

    if n > 0:
        pval = binomtest(k=better, n=n, p=0.5, alternative="two-sided").pvalue
    else:
        pval = np.nan

    stat_rows.append({
        "scope": label,
        "n_usable_groups": n,
        "hybrid_better": better,
        "hybrid_worse": worse,
        "p_value_two_sided": pval
    })

pd.DataFrame(stat_rows).to_csv(os.path.join(OUTPUT_DIR, "groupwise_exact_sign_test.csv"), index=False)

##Cell 24 — Final file list

In [32]:
print("Files written to:", OUTPUT_DIR)
print(sorted(os.listdir(OUTPUT_DIR)))

Files written to: /content/rebuild_outputs
['baseline_group_ranking_results.csv', 'dataset_card_stage3.txt', 'dataset_schema_summary.csv', 'feature_filtering_audit.csv', 'full_feature_candidates_before_split.txt', 'full_model_feature_list.csv', 'full_model_holdout_predictions.csv', 'group_ranking_metrics_all_and_comparable.csv', 'group_ranking_uncertainty.csv', 'groupwise_exact_sign_test.csv', 'heuristic_group_ranking_metrics.csv', 'heuristic_group_ranking_results.csv', 'holdout_classification_metrics.csv', 'hybrid_group_ranking_results.csv', 'leakage_excluded_columns.txt', 'pairwise_dependency_audit.txt', 'pairwise_model_feature_list.csv', 'pairwise_model_holdout_comparisons.csv', 'phase5_missing_group_key_audit.csv', 'phase5_reconciliation_audit.csv', 'random_group_ranking_metrics.csv', 'random_group_ranking_repeat_summary.csv', 'random_group_ranking_results_all_repeats.csv', 'sanity_baseline_comparison.csv', 'split_audit_summary.csv', 'test_groups_detailed.csv', 'test_groups_missing

## Revision addendum v1.1 — feature importance, paired effect sizes, and aligned tables

This addendum was added for the revised manuscript and Supporting Information. It generates the additional reproducibility outputs introduced during peer-review revision:

- permutation feature importance for the leakage-clean baseline model;
- paired ranking effect sizes and bootstrap confidence intervals;
- aligned group-ranking and sanity-baseline tables used in the revised SI;
- corrected uncertainty and exact sign-test summaries.

The addendum assumes the earlier leakage-clean pipeline cells have already produced the final selected feature list and group-ranking outputs in `OUTPUT_DIR`. When running outside Colab, adjust `INPUT_STAGE3` and `OUTPUT_DIR` as needed.

In [ ]:
# ============================================================
# Revision addendum v1.1
# Additional analyses for revised manuscript/SI
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from scipy.stats import binomtest

REVISION_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "revision_outputs")
os.makedirs(REVISION_OUTPUT_DIR, exist_ok=True)

# ---------- Helpers ----------
def assign_feature_family(feature):
    if feature.startswith("analyte_"):
        return "Analyte descriptor"
    if feature.startswith("hba_"):
        return "HBA descriptor"
    if feature.startswith("hbd_"):
        return "HBD descriptor"
    if feature.startswith("des_"):
        return "DES-level descriptor"
    if feature.startswith("pair_"):
        return "DES-analyte interaction"
    if feature.startswith("ratio_"):
        return "Composition / ratio"
    return "Other"


def bootstrap_mean_ci(values, n_boot=10000, random_state=42):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(random_state)
    boot = np.array([
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_boot)
    ])
    return np.quantile(boot, 0.025), np.quantile(boot, 0.975)


def compute_group_metrics(rank_df, rank_col):
    rows = []
    for (doi, analyte), g in rank_df.groupby(["doi", "analyte"], dropna=False):
        g = g.copy()
        n = len(g)
        opt_ranks = g.loc[g["is_optimal"].astype(int) == 1, rank_col].tolist()
        if len(opt_ranks) == 0:
            continue
        best_rank = min(opt_ranks)
        rows.append({
            "doi": doi,
            "analyte": analyte,
            "group_size": n,
            "Hit@1": int(best_rank == 1),
            "Hit@3": int(best_rank <= 3),
            "MRR": 1.0 / best_rank,
        })
    return pd.DataFrame(rows)


def summarize_scope(df_scope, scope_name):
    delta_hit1 = df_scope["Hit@1_hybrid"].values - df_scope["Hit@1_baseline"].values
    delta_mrr = df_scope["MRR_hybrid"].values - df_scope["MRR_baseline"].values
    better = int((delta_mrr > 0).sum())
    worse = int((delta_mrr < 0).sum())
    equal = int((delta_mrr == 0).sum())
    n_non_tied = better + worse
    p_value = binomtest(better, n=n_non_tied, p=0.5, alternative="two-sided").pvalue if n_non_tied > 0 else np.nan
    hit1_ci_low, hit1_ci_high = bootstrap_mean_ci(delta_hit1)
    mrr_ci_low, mrr_ci_high = bootstrap_mean_ci(delta_mrr)
    dz_mrr = float(np.mean(delta_mrr) / np.std(delta_mrr, ddof=1)) if len(delta_mrr) > 1 and np.std(delta_mrr, ddof=1) > 0 else np.nan
    return {
        "scope": scope_name,
        "n_groups": len(df_scope),
        "baseline_Hit@1_mean": df_scope["Hit@1_baseline"].mean(),
        "hybrid_Hit@1_mean": df_scope["Hit@1_hybrid"].mean(),
        "delta_Hit@1_hybrid_minus_baseline": np.mean(delta_hit1),
        "delta_Hit@1_bootstrap_CI_low": hit1_ci_low,
        "delta_Hit@1_bootstrap_CI_high": hit1_ci_high,
        "baseline_MRR_mean": df_scope["MRR_baseline"].mean(),
        "hybrid_MRR_mean": df_scope["MRR_hybrid"].mean(),
        "delta_MRR_hybrid_minus_baseline": np.mean(delta_mrr),
        "delta_MRR_bootstrap_CI_low": mrr_ci_low,
        "delta_MRR_bootstrap_CI_high": mrr_ci_high,
        "paired_standardized_effect_size_dz_MRR": dz_mrr,
        "hybrid_better_by_MRR": better,
        "hybrid_worse_by_MRR": worse,
        "equal_by_MRR": equal,
        "exact_sign_test_p_value_ties_excluded": p_value,
    }

# ---------- Permutation feature importance ----------
selected_features_path = os.path.join(OUTPUT_DIR, "train_selected_features_after_filtering.txt")
if not os.path.exists(selected_features_path):
    selected_features_path = "train_selected_features_after_filtering.txt"

with open(selected_features_path, "r", encoding="utf-8") as f:
    selected_features = [line.strip() for line in f if line.strip()]

X_train = df_train[selected_features].copy()
X_test = df_test[selected_features].copy()
y_train = df_train["is_optimal"].astype(int).values
y_test = df_test["is_optimal"].astype(int).values

full_model_for_importance = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2", C=1.0, class_weight="balanced",
        solver="lbfgs", max_iter=5000, random_state=RANDOM_STATE
    )),
])
full_model_for_importance.fit(X_train, y_train)

perm_auc = permutation_importance(
    full_model_for_importance, X_test, y_test,
    scoring="roc_auc", n_repeats=50, random_state=RANDOM_STATE, n_jobs=1
)
perm_bacc = permutation_importance(
    full_model_for_importance, X_test, y_test,
    scoring="balanced_accuracy", n_repeats=50, random_state=RANDOM_STATE, n_jobs=1
)

coef = full_model_for_importance.named_steps["clf"].coef_[0]
feature_importance_df = pd.DataFrame({
    "feature": selected_features,
    "permutation_auc_mean_decrease": perm_auc.importances_mean,
    "permutation_auc_sd": perm_auc.importances_std,
    "permutation_balanced_accuracy_mean_decrease": perm_bacc.importances_mean,
    "permutation_balanced_accuracy_sd": perm_bacc.importances_std,
    "standardized_logistic_coefficient": coef,
    "abs_standardized_logistic_coefficient": np.abs(coef),
})
feature_importance_df["feature_family"] = feature_importance_df["feature"].apply(assign_feature_family)
feature_importance_df = feature_importance_df.sort_values("permutation_auc_mean_decrease", ascending=False).reset_index(drop=True)
feature_importance_df.to_csv(os.path.join(REVISION_OUTPUT_DIR, "feature_importance_permutation_full_model.csv"), index=False)
feature_importance_df.head(20).to_csv(os.path.join(REVISION_OUTPUT_DIR, "feature_importance_top20.csv"), index=False)

family_summary = (
    feature_importance_df.groupby("feature_family", dropna=False)
    .agg(
        n_features=("feature", "count"),
        mean_auc_decrease=("permutation_auc_mean_decrease", "mean"),
        max_auc_decrease=("permutation_auc_mean_decrease", "max"),
        n_positive_auc_importance=("permutation_auc_mean_decrease", lambda x: int((x > 0).sum())),
    )
    .reset_index()
    .sort_values("mean_auc_decrease", ascending=False)
)
family_summary.to_csv(os.path.join(REVISION_OUTPUT_DIR, "feature_importance_family_summary.csv"), index=False)

# ---------- Paired ranking effect sizes ----------
baseline_rank_df = pd.read_csv(os.path.join(OUTPUT_DIR, "baseline_group_ranking_results.csv"))
hybrid_rank_df = pd.read_csv(os.path.join(OUTPUT_DIR, "hybrid_group_ranking_results.csv"))
base_group = compute_group_metrics(baseline_rank_df, "rank_baseline")
hyb_group = compute_group_metrics(hybrid_rank_df, "rank_hybrid")
paired = base_group.merge(hyb_group, on=["doi", "analyte", "group_size"], suffixes=("_baseline", "_hybrid"))

effect_rows = [
    summarize_scope(paired, "all_groups"),
    summarize_scope(paired.loc[paired["group_size"] >= 2], "comparable_ge_2"),
    summarize_scope(paired.loc[paired["group_size"] >= 3], "comparable_ge_3"),
]
effect_df = pd.DataFrame(effect_rows)
effect_df.to_csv(os.path.join(REVISION_OUTPUT_DIR, "paired_ranking_effect_sizes.csv"), index=False)

main_table4 = effect_df[[
    "scope", "n_groups", "baseline_Hit@1_mean", "hybrid_Hit@1_mean",
    "baseline_MRR_mean", "hybrid_MRR_mean", "delta_MRR_hybrid_minus_baseline",
    "hybrid_better_by_MRR", "hybrid_worse_by_MRR", "equal_by_MRR",
    "exact_sign_test_p_value_ties_excluded",
]].copy()
main_table4.to_csv(os.path.join(REVISION_OUTPUT_DIR, "main_table4_revised_group_ranking.csv"), index=False)

print("Revision addendum outputs written to:", REVISION_OUTPUT_DIR)
print("Top feature importance rows:")
print(feature_importance_df.head(10).to_string(index=False))
print("\nPaired effect-size summary:")
print(effect_df.to_string(index=False))